In [1]:
%%capture
!pip install -q -U "transformers>=4.46" accelerate peft bitsandbytes trl datasets scikit-learn joblib hf_transfer


In [8]:
import os
import json
import random
import zipfile

import numpy as np
import torch
import transformers
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from trl import SFTTrainer, SFTConfig
from datasets import Dataset

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
transformers.set_seed(SEED)

MODEL_NAME = "Qwen3-4B-Instruct-2507"


In [3]:
DATA_DIR = "/kaggle/input/datasets/kunimsmk/scolar-red/"

kid_adult_path = os.path.join(DATA_DIR, "kid_adult.jsonl")
public_test_style_path = os.path.join(DATA_DIR, "public_test_style.jsonl")
style_clf_path = os.path.join(DATA_DIR, "style_clf.pkl")

def load_jsonl(path):
    records = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                records.append(json.loads(line))
    return records

kid_adult = load_jsonl(kid_adult_path)
public_test_style = load_jsonl(public_test_style_path)


In [9]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

def format_example(record):
    messages = [
        {"role": "user", "content": record["prompt"]},
        {"role": "assistant", "content": record["kid"]},
    ]
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=False)
    return {"text": text}

sft_records = [format_example(r) for r in kid_adult]
train_dataset = Dataset.from_list(sft_records)


In [10]:
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.float16,
)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto",
    torch_dtype=torch.float16,
    trust_remote_code=True,
)
model.config.use_cache = False

model = prepare_model_for_kbit_training(model)

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
)
model = get_peft_model(model, lora_config)


[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

In [ ]:
sft_config = SFTConfig(
    output_dir="./sft_out",
    per_device_train_batch_size=2,
    gradient_accumulation_steps=8,
    num_train_epochs=3,
    learning_rate=2e-4,
    lr_scheduler_type="cosine",
    warmup_ratio=0.03,
    logging_steps=10,
    save_strategy="no",
    fp16=False,
    bf16=False,
    optim="paged_adamw_8bit",
    seed=SEED,
    data_seed=SEED,
    report_to=[],
    dataset_text_field="text",
    packing=False,
    loss_type="nll"
)

trainer = SFTTrainer(
    model=model,
    args=sft_config,
    train_dataset=train_dataset,
    processing_class=tokenizer,
)

trainer.train()


In [ ]:
ADAPTER_DIR = "./sft_adapter"
trainer.model.save_pretrained(ADAPTER_DIR)
tokenizer.save_pretrained(ADAPTER_DIR)


In [ ]:
model.eval()
model.config.use_cache = True

def generate_response(prompt_text, max_new_tokens=256):
    messages = [{"role": "user", "content": prompt_text}]
    inputs = tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        return_tensors="pt",
        return_dict=True,
    ).to(model.device)
    input_len = inputs["input_ids"].shape[-1]
    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            num_beams=1,
            temperature=None,
            top_p=None,
            top_k=None,
            pad_token_id=tokenizer.pad_token_id,
        )
    new_tokens = output_ids[0][input_len:]
    return tokenizer.decode(new_tokens, skip_special_tokens=True)

test_prompts = [r["prompt"] for r in public_test_style]
generated_responses = [generate_response(p) for p in test_prompts]


In [ ]:
import joblib
from scipy.sparse import hstack

style_clf = joblib.load(style_clf_path)
vec1, vec2 = style_clf["vecs"]
estimator = style_clf["clf"]

X = hstack([vec1.transform(generated_responses), vec2.transform(generated_responses)]).tocsr()
p_simple_scores = estimator.predict_proba(X)[:, 1]
p_simple_mean = float(np.mean(p_simple_scores))

print(f"P_simple: {p_simple_mean}")


In [11]:
from trl import DPOTrainer, DPOConfig

def build_dpo_example(record):
    prompt_text = tokenizer.apply_chat_template(
        [{"role": "user", "content": record["prompt"]}],
        tokenize=False,
        add_generation_prompt=True,
    )
    return {"prompt": prompt_text, "chosen": record["kid"], "rejected": record["adult"]}

dpo_dataset = Dataset.from_list([build_dpo_example(r) for r in kid_adult])


In [13]:
model.config.use_cache = False

dpo_config = DPOConfig(
    output_dir="./dpo_style_out",
    per_device_train_batch_size=2,
    gradient_accumulation_steps=8,
    num_train_epochs=3,
    learning_rate=5e-5,
    lr_scheduler_type="cosine",
    warmup_ratio=0.03,
    logging_steps=10,
    save_strategy="no",
    fp16=False,
    bf16=False,
    optim="paged_adamw_8bit",
    beta=0.1,
    seed=SEED,
    data_seed=SEED,
    report_to=[],
)

dpo_trainer = DPOTrainer(
    model=model,
    args=dpo_config,
    train_dataset=dpo_dataset,
    processing_class=tokenizer,
)

dpo_trainer.train()


NameError: name 'RewardConfig' is not defined

In [ ]:
DPO_STYLE_ADAPTER_DIR = "./dpo_style_adapter"
dpo_trainer.model.save_pretrained(DPO_STYLE_ADAPTER_DIR)
tokenizer.save_pretrained(DPO_STYLE_ADAPTER_DIR)


In [ ]:
model.eval()
model.config.use_cache = True

generated_responses = [generate_response(p) for p in test_prompts]

X = hstack([vec1.transform(generated_responses), vec2.transform(generated_responses)]).tocsr()
p_simple_scores = estimator.predict_proba(X)[:, 1]
p_simple_mean = float(np.mean(p_simple_scores))

print(f"P_simple: {p_simple_mean:}")

In [14]:
from transformers import AutoModelForSequenceClassification
from trl import RewardTrainer, RewardConfig

good_bad_path = os.path.join(DATA_DIR, "good_bad.jsonl")
public_test_quality_path = os.path.join(DATA_DIR, "public_test_quality.jsonl")

good_bad = load_jsonl(good_bad_path)
public_test_quality = load_jsonl(public_test_quality_path)


In [15]:
def build_reward_example(record):
    chosen_text = tokenizer.apply_chat_template(
        [{"role": "user", "content": record["instruction"]}, {"role": "assistant", "content": record["chosen"]}],
        tokenize=False,
    )
    rejected_text = tokenizer.apply_chat_template(
        [{"role": "user", "content": record["instruction"]}, {"role": "assistant", "content": record["rejected"]}],
        tokenize=False,
    )
    return {"chosen": chosen_text, "rejected": rejected_text}

reward_dataset = Dataset.from_list([build_reward_example(r) for r in good_bad])


In [19]:
rm_model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=1,
    quantization_config=bnb_config,
    device_map="auto",
    torch_dtype=torch.float16,
    trust_remote_code=True,
)
rm_model.config.use_cache = False
rm_model.config.pad_token_id = tokenizer.pad_token_id

rm_model = prepare_model_for_kbit_training(rm_model)

rm_lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="SEQ_CLS",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    modules_to_save=["score"],
)
rm_model = get_peft_model(rm_model, rm_lora_config)

for param in rm_model.parameters():
    if param.dtype == torch.bfloat16:
        param.data = param.data.to(torch.float32)

for buf in rm_model.buffers():
    if buf.dtype == torch.bfloat16:
        buf.data = buf.data.to(torch.float32)
for name, module in rm_model.named_modules():
    if name.endswith("score") and hasattr(module, "weight"):
        module.to(dtype=torch.float32)

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

[transformers] Qwen3ForSequenceClassification LOAD REPORT from: /kaggle/input/models/yngbogdnn27/qwen3-4b-instruct-2507/pytorch/default/2
Key          | Status  | 
-------------+---------+-
score.weight | MISSING | 

Notes:
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [22]:
reward_config = RewardConfig(
    output_dir="./reward_model_out",
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,
    num_train_epochs=2,
    learning_rate=1e-4,
    lr_scheduler_type="cosine",
    warmup_ratio=0.03,
    logging_steps=10,
    save_strategy="no",
    fp16=False,
    bf16=False,
    optim="paged_adamw_8bit",
    seed=SEED,
    data_seed=SEED,
    report_to=[],
)


reward_trainer = RewardTrainer(
    model=rm_model,
    args=reward_config,
    train_dataset=reward_dataset,
    processing_class=tokenizer,
)

reward_trainer.train()


[transformers] warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


Adding EOS to train dataset:   0%|          | 0/2226 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/2226 [00:00<?, ? examples/s]

Filtering train >1024 tokens:   0%|          | 0/2226 [00:00<?, ? examples/s]

Step,Training Loss
10,0.281356
20,0.113231
30,0.019380
40,0.073331
50,0.015974
60,0.033048
70,0.010904
80,0.002195
90,0.078501
100,0.023677


TrainOutput(global_step=280, training_loss=0.029042257549571918, metrics={'train_runtime': 4295.9987, 'train_samples_per_second': 1.036, 'train_steps_per_second': 0.065, 'total_flos': 4.3643937954816e+16, 'train_loss': 0.029042257549571918, 'epoch': 2.0})

In [23]:
REWARD_ADAPTER_DIR = "./reward_model_adapter"
reward_trainer.model.save_pretrained(REWARD_ADAPTER_DIR)
tokenizer.save_pretrained(REWARD_ADAPTER_DIR)


('./reward_model_adapter/tokenizer_config.json',
 './reward_model_adapter/chat_template.jinja',
 './reward_model_adapter/tokenizer.json')

In [26]:
rm_model.eval()

def get_reward_score(text):
    inputs = tokenizer(text, return_tensors="pt", truncation=True, max_length=1024).to(rm_model.device)
    with torch.no_grad():
        logits = rm_model(**inputs).logits
    return logits.item()

wins = 0
for r in public_test_quality:
    chosen_text = tokenizer.apply_chat_template(
        [{"role": "user", "content": r["prompt"]}, {"role": "assistant", "content": r["chosen"]}],
        tokenize=False,
    )
    rejected_text = tokenizer.apply_chat_template(
        [{"role": "user", "content": r["prompt"]}, {"role": "assistant", "content": r["rejected"]}],
        tokenize=False,
    )
    if get_reward_score(chosen_text) > get_reward_score(rejected_text):
        wins += 1

pairwise_accuracy = wins / len(public_test_quality)

print(f"Pairwise accuracy: {pairwise_accuracy}")


Pairwise accuracy (public_test_quality, n=50): 0.9400
